# CASCADES v9 — 1-Click Quickstart

> **Goal:** Load the pre-trained `cascades_v9_weights.pt`, inject `CASCADESLinear` adapters into a frozen `Qwen3-4B` base model (4-bit), and chat with it using **Subspace-Contrastive Decoding** (adapter-level CFG boost) to activate its trained reasoning patterns.

**No training required.** This notebook downloads the pre-trained weights and runs inference only.

**Requirements:**
- Google Colab with an **A100 / T4 / L4** GPU runtime (`Runtime → Change runtime type → GPU`)
- ~5.5 GB GPU VRAM (A100 40GB is ideal; T4 16GB also works)

---
| Section | What it does |
|---|---|
| 1. Setup | Clone repo, install dependencies |
| 2. Load Model | Qwen3-4B in NF4 4-bit |
| 3. Inject Adapters | `CASCADESLinear` wrappers |
| 4. Load Weights | Download `cascades_v9_weights.pt` from HuggingFace |
| 5. Chat | Interactive reasoning with CFG boost |

## 1. Setup — Clone Repo & Install Dependencies

This takes ~2 minutes on first run.

In [ ]:
# Clone the CASCADES repository
import os
if not os.path.exists("/content/CASCADES"):
    !git clone https://github.com/YOUR_USERNAME/CASCADES--continual-PEFT-for-Local-LLMs.git /content/CASCADES
    print("Repository cloned.")
else:
    print("Repository already present, pulling latest.")
    !git -C /content/CASCADES pull

%cd /content/CASCADES

# Install Python dependencies
# bitsandbytes is pinned to the Colab-compatible wheel
!pip install -q \
    torch>=2.3.0 \
    transformers>=4.44.0 \
    accelerate>=0.33.0 \
    bitsandbytes>=0.43.0 \
    tokenizers>=0.21.1 \
    datasets>=2.20.0 \
    rich>=13.0.0

print("\nAll dependencies installed.")

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU detected!\n"
        "Please enable a GPU runtime: Runtime → Change runtime type → GPU"
    )

gpu_name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1024**3
print(f"GPU: {gpu_name}  |  VRAM: {vram_gb:.1f} GB")

if vram_gb < 12:
    print("WARNING: Less than 12 GB VRAM. T4 (16 GB) is minimum recommended.")
else:
    print("VRAM OK.")

## 2. Load Qwen3-4B in 4-bit (NF4)

The base model is frozen. Only the CASCADES adapters (injected in step 3) are active.

This downloads ~2.4 GB on first run and takes ~90 seconds.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

MODEL_ID = "p-e-w/Qwen3-4B-Instruct-2507-heretic"
print(f"Loading: {MODEL_ID}")

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)

vram_used = torch.cuda.memory_allocated(0) / 1024**3
print(f"\nModel loaded. VRAM used: {vram_used:.2f} GB")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e9:.2f}B total (all frozen)")

## 3. Inject CASCADESLinear Adapters

This wraps the target projection layers (`q_proj`, `v_proj`, `up_proj`, `down_proj`)
with `CASCADESLinear` — which adds either a full `CASCADESAdapter` (critical layers)
or a lightweight `FunLoRA` rank-1 adapter (non-critical layers), without touching
the frozen base weights.

In [ ]:
import sys
sys.path.insert(0, "/content/CASCADES")

from cascades.injection import inject_cascades, estimate_quant_noise
from cascades.config import DEFAULT_CONFIG

# Estimate quantization noise for DEAL filter
quant_noise = estimate_quant_noise(model)
print(f"Quantization noise floor: {quant_noise:.6f}")

# Inject adapters (rank=8, all components enabled)
critical_adapters, funlora_adapters = inject_cascades(
    model,
    rank=8,
    config=DEFAULT_CONFIG,
)

# Propagate the quantization noise to each adapter
for a in critical_adapters:
    a.quant_noise_std.fill_(quant_noise)

# Count trainable parameters (adapter-only)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nInjected adapters:")
print(f"  Critical (full CASCADESAdapter) : {len(critical_adapters)}")
print(f"  Non-critical (FunLoRA rank-1)   : {len(funlora_adapters)}")
print(f"  Trainable parameters            : {trainable / 1e6:.2f}M / {total / 1e6:.0f}M total")
print(f"  Adapter overhead                : {trainable / total * 100:.2f}%")

## 4. Download Pre-trained Weights

Downloads the `cascades_v9_weights.pt` checkpoint from HuggingFace.
This contains the trained adapter state for the 3-task reasoning sequence
(GSM8K → ARC → CommonsenseQA) that achieves the 46.82% Proxy Accuracy
reported in Table 1 of the v9 paper.

In [ ]:
import os

WEIGHTS_PATH = "/content/CASCADES/cascades_v9_weights.pt"
HF_WEIGHTS_URL = "https://huggingface.co/YOUR_HF_USERNAME/cascades-v9/resolve/main/cascades_v9_weights.pt"

if not os.path.exists(WEIGHTS_PATH):
    print(f"Downloading pre-trained weights from HuggingFace...")
    !wget -q --show-progress -O "{WEIGHTS_PATH}" "{HF_WEIGHTS_URL}"
    print("Download complete.")
else:
    size_mb = os.path.getsize(WEIGHTS_PATH) / 1024**2
    print(f"Weights already present: {WEIGHTS_PATH} ({size_mb:.1f} MB)")

# Load the checkpoint into the injected adapters
checkpoint = torch.load(WEIGHTS_PATH, map_location="cpu", weights_only=True)

# The checkpoint stores adapter state_dicts keyed by adapter index
missing, unexpected = model.load_state_dict(checkpoint, strict=False)

# Only adapter parameters should be present in the checkpoint;
# missing base-model keys are expected (they are frozen and not saved)
adapter_missing = [k for k in missing if "adapter" in k or "cascades" in k.lower()]
if adapter_missing:
    print(f"WARNING: {len(adapter_missing)} adapter keys not found in checkpoint:")
    for k in adapter_missing[:5]:
        print(f"  {k}")
else:
    print(f"Weights loaded successfully.")
    print(f"  Loaded keys : {len(checkpoint)}")
    print(f"  Skipped keys: {len(missing)} (expected: frozen base model weights)")

## 5. Interactive Chat with Subspace-Contrastive Decoding

**What is CFG boost?**

The Heretic model defaults to conversational responses ("Certainly! Here is...") even
after fine-tuning on `<think>` CoT format. To activate its trained reasoning patterns,
CASCADES uses **Subspace-Contrastive Decoding (SCD)** — adapter-level Classifier-Free
Guidance that amplifies the adapter contribution relative to the base model's prior:

```
output = base_out + λ_cfg × adapter_out
```

`λ_cfg = 1.0` is neutral (standard LoRA). `λ_cfg = 1.5–2.0` steers toward CoT reasoning.

Run the cell below and type your question when prompted.

In [ ]:
import torch
from cascades.adapters import CASCADESLinear


def set_cfg_lambda(model, lambda_cfg: float) -> int:
    """Set the CFG boost strength on all CASCADESLinear wrappers.

    This patches the forward_with_cfg lambda without modifying weights.
    Call this before generation to activate Subspace-Contrastive Decoding.

    Args:
        model: The CASCADES-adapted model.
        lambda_cfg: Boost strength. 1.0 = neutral, 1.5 = recommended, 2.0 = aggressive.

    Returns:
        Number of adapters updated.
    """
    count = 0
    for module in model.modules():
        if isinstance(module, CASCADESLinear):
            module._cfg_lambda = lambda_cfg
            count += 1
    return count


def patch_forward_with_cfg(model, lambda_cfg: float = 1.5) -> None:
    """Monkey-patch CASCADESLinear.forward to use CFG boost during inference.

    Replaces the standard forward (base + 0.1 * adapter) with the CFG variant
    (base + 0.1 * lambda_cfg * adapter). This is done via patching rather than
    modifying the class to allow easy reversion.

    Args:
        model: The CASCADES-adapted model.
        lambda_cfg: CFG strength to apply.
    """
    import functools
    import types

    for module in model.modules():
        if isinstance(module, CASCADESLinear):
            # Store the original forward for potential restoration
            if not hasattr(module, "_original_forward"):
                module._original_forward = module.forward

            lam = lambda_cfg

            def _cfg_forward(self, x, *args, _lam=lam, **kwargs):
                """CFG-boosted forward pass."""
                base_out = (
                    self.base_layer(x, *args, **kwargs)
                    if not isinstance(self.base_layer, torch.nn.Linear)
                    else self.base_layer(x)
                )
                if self.is_critical and hasattr(self.adapter, "cfg_boost"):
                    adapt_out = self.adapter.cfg_boost(x, lambda_cfg=_lam)
                else:
                    adapt_out = self.adapter(x)
                return base_out + 0.1 * adapt_out.to(base_out.dtype)

            module.forward = types.MethodType(_cfg_forward, module)


def restore_standard_forward(model) -> None:
    """Restore the original forward pass (disable CFG boost)."""
    for module in model.modules():
        if isinstance(module, CASCADESLinear) and hasattr(module, "_original_forward"):
            module.forward = module._original_forward


print("CFG helper functions defined.")

In [ ]:
SYSTEM_PROMPT = (
    "You are a precise reasoning assistant. When solving problems:\n"
    "1. Think step-by-step inside <think>...</think> tags\n"
    "2. After </think>, output ONLY the final answer on a new line\n"
    "3. The final answer should be concise (a number, expression, or short phrase)\n"
    "4. Do NOT add explanations after </think>\n\n"
    "Example:\n"
    "<think>\n[your reasoning steps]\n</think>\n\n42"
)


@torch.inference_mode()
def generate_response(
    question: str,
    lambda_cfg: float = 1.5,
    max_new_tokens: int = 512,
    temperature: float = 0.1,
) -> str:
    """Generate a reasoned response using Subspace-Contrastive Decoding.

    Args:
        question: The user's question or reasoning problem.
        lambda_cfg: CFG boost strength (1.0 = neutral, 1.5 = recommended).
        max_new_tokens: Maximum tokens to generate.
        temperature: Sampling temperature (0.0 = greedy, 0.1 = near-greedy).

    Returns:
        The model's generated response string.
    """
    model.eval()

    # Activate CFG boost for this generation
    patch_forward_with_cfg(model, lambda_cfg=lambda_cfg)

    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": question},
    ]

    try:
        prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        # Fallback for tokenizers that don't support system role
        combined_content = f"{SYSTEM_PROMPT}\n\n{question}"
        messages = [{"role": "user", "content": combined_content}]
        prompt_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )

    inputs = tokenizer(prompt_text, return_tensors="pt").to("cuda")

    model.config.use_cache = True
    try:
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            do_sample=(temperature > 0),
            pad_token_id=tokenizer.eos_token_id,
        )
    finally:
        # Always restore standard forward after generation
        restore_standard_forward(model)

    response = tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

    return response


print("Generation helper ready. Run the chat cell below.")

In [ ]:
# ============================================================
# INTERACTIVE CHAT — Edit the question and run this cell
# ============================================================

# CFG boost strength: 1.0 = neutral LoRA, 1.5 = recommended, 2.0 = aggressive
CFG_LAMBDA = 1.5

# Your question here:
QUESTION = """Janet's ducks lay 16 eggs per day. She eats three for breakfast every morning
and bakes muffins for her friends every day with four. She sells the remainder at the
farmers' market daily for $2 per fresh duck egg. How much in dollars does she make
every day at the farmers' market?"""

print(f"[CFG λ = {CFG_LAMBDA}]")
print(f"Question: {QUESTION.strip()}")
print("-" * 60)

response = generate_response(QUESTION, lambda_cfg=CFG_LAMBDA)

print("Response:")
print(response)
print("-" * 60)

# VRAM snapshot
used = torch.cuda.memory_allocated(0) / 1024**3
peak = torch.cuda.max_memory_allocated(0) / 1024**3
print(f"VRAM: {used:.2f} GB used | {peak:.2f} GB peak")

In [ ]:
# ============================================================
# CFG ABLATION — Compare λ=1.0 (neutral) vs λ=1.5 (boosted)
# ============================================================
# This demonstrates the effect of Subspace-Contrastive Decoding.
# The base model (λ=1.0) often defaults to conversational output.
# With λ=1.5, the adapter steers it toward structured CoT reasoning.

ABLATION_QUESTION = """If a train travels at 60 mph for 2.5 hours,
then at 80 mph for another 1.5 hours, what is the total distance travelled?"""

print("=" * 60)
print("CFG ABLATION: Same question, two decoding modes")
print("=" * 60)

for lam in [1.0, 1.5]:
    label = "Neutral (standard LoRA-like)" if lam == 1.0 else "CFG Boosted (SCD active)"
    print(f"\n[λ = {lam}] {label}")
    print("-" * 40)
    resp = generate_response(ABLATION_QUESTION, lambda_cfg=lam, max_new_tokens=256)
    print(resp)

print("\n" + "=" * 60)

## Notes

### Expected Behaviour

- **With λ=1.0:** The Heretic model often responds conversationally ("Sure! Let me help...") — this is the base model's prior dominating.
- **With λ=1.5:** The adapter contribution is amplified, steering the model toward `<think>` CoT format and concise final answers.
- **EM = 0%:** As documented in §6.1 footnote ¹ of the paper, Exact Match is near 0% because the model generates in conversational format rather than adhering to rigid `<think>` syntax during unconstrained inference. The **Proxy Accuracy (exp(−loss) = 46.82%)** confirms that the mathematical and logic transfer occurred without catastrophic forgetting.

### Architecture Summary

```
Frozen Qwen3-4B (NF4)
   └── CASCADESLinear (wrapper per target layer)
          ├── base_layer (frozen Linear4bit)
          └── adapter
               ├── CASCADESAdapter (critical layers)
               │    ├── U_shared, V_shared  ← Stiefel manifold basis
               │    └── ResonantCore        ← Hebbian-routed core pool (K=4 cores)
               └── FunLoRA_Adapter (non-critical layers)
                    └── rank-1 outer product with fused nonlinear activation
```

### References

- [CASCADES v9 Paper](papers/CASCADES_v9_Final_Paper.md)
- [Plain Language Summary](papers/CASCADES_v9_Plain_Language.md)
- [Audit Report](AUDIT_REPORT.md)